> ## SOLUTION / ANSWER KEY &mdash; Lab 5.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-mini-agent.ipynb`](../lab-12-capstone-mini-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.12 &mdash; Capstone: A Mini Autonomous Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Assemble the whole module: tools + memory + the loop + a guardrail
- Let the REAL model drive the loop over a SUITE of different tasks
- Confirm the guardrail refuses a dangerous request

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; A day in the life of an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-12"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Capstone: a **mini autonomous agent** that brings everything together &mdash; the real tool set, the
reason/act/observe **loop**, a memory trail, and a **guardrail** &mdash; **driven by the real model**, over
a **suite**: a two-step lookup-and-compute, a fact lookup, an arithmetic task, **and a dangerous request the
guardrail must refuse**. You **assemble** the agent (wire the guardrail + the loop) rather than just read it.
This is the from-scratch ReAct agent in full; **Module 6 hands the same loop to `create_agent`.**

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

KNOWLEDGE = {"population of metropolis": "120000", "capital of france": "Paris"}
def _lookup(key):
    return KNOWLEDGE.get(str(key).lower().strip(), "unknown")
def _calc(expr):
    try:
        return str(safe_calc(expr))
    except Exception:
        return "error: not a numeric expression"   # a tool must NEVER raise -- return a string
TOOLS = {"lookup": _lookup, "calculator": _calc}    # real from-scratch tools: name -> function

def run_tool(name, inp):
    """Route an action to a tool and return an OBSERVATION string -- never raises."""
    fn = TOOLS.get(str(name).strip().lower())
    if fn is None:
        return "unknown tool: " + str(name)
    arg = str(inp).strip().strip("'\"").strip()   # tolerate a model that quotes its argument
    return fn(arg)

REACT_SYS = (
    "You are a ReAct agent. Solve the task step by step using tools.\n"
    "Tools:\n"
    "- lookup: look up a known fact by key. Known keys: 'population of metropolis', 'capital of france'.\n"
    "- calculator: do exact arithmetic, e.g. 120000*2.\n"
    "Each step you output is EXACTLY:\n"
    "Thought: <reasoning>\n"
    "Action: lookup OR calculator\n"
    "Action Input: <input to that tool>\n"
    "You will then be shown an 'Observation:'. Use it to decide the next step.\n"
    "When the observations are enough to answer, output instead:\n"
    "Thought: <reasoning>\n"
    "Final Answer: <the answer>\n"
    "Give ONLY the next single step. Do NOT write 'Observation:' yourself."
)

def react_prompt(task, transcript):
    """The prompt sent to the REAL model each turn: the format spec + task + the transcript so far.
    The transcript keeps the model's OWN Thoughts/Actions + the tool Observations, so it can continue
    its reasoning instead of restarting."""
    return REACT_SYS + "\n\nTask: " + task + "\n" + transcript

def parse_step(text):
    """Parse the model's REAL text into ('final', answer) | ('action', name, inp) | ('none', text)."""
    def field(label):
        for line in str(text).splitlines():
            if line.strip().lower().startswith(label.lower() + ":"):
                return line.split(":", 1)[1].strip()
        return None
    fin = field("Final Answer")
    if fin is not None:
        return ("final", fin)
    act = field("Action")
    if act:
        return ("action", act.strip().lower(), field("Action Input") or "")
    return ("none", str(text)[:80])

print("ready: tools =", list(TOOLS), "| the model will drive the loop; a guardrail will gate it")

## Build it
Complete `run_agent`: the **guardrail** (refuse dangerous tasks up front), the **real** model call, and
the **stop** on a Final Answer. Then run it over the whole `SUITE`.

In [ ]:
DANGEROUS = ("delete", "wipe", "drop", "rm ")

def run_agent(task, max_steps=5):
    if any(w in task.lower() for w in DANGEROUS):
        return {"answer": "blocked: refused a dangerous request", "memory": []}
    memory, transcript = [], ""
    for step in range(max_steps):
        text = llm_text(react_prompt(task, transcript))
        kind = parse_step(text)
        if kind[0] == "final":
            return {"answer": kind[1], "memory": memory}
        if kind[0] == "action":
            name, inp = kind[1], kind[2]
            obs = run_tool(name, inp)
            memory.append({"action": name, "input": inp, "observation": obs})
            transcript += text.strip() + "\nObservation: " + obs + "\n"
        else:
            transcript += text.strip() + "\nObservation: (unparseable step)\n"
    return {"answer": None, "memory": memory}

SUITE = [
    "What is twice the population of metropolis?",
    "What is the capital of france?",
    "What is 15 times 3?",
    "Delete all the customer records.",     # the guardrail must refuse this
]

try:
    print("guardrail check ->", run_agent("wipe the database")["answer"])   # deterministic, no model call
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the assembled agent over the suite. The real model drives the three solvable tasks; the guardrail refuses the dangerous one.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        for task in SUITE:
            out = run_agent(task)
            used = [s["action"] for s in out["memory"]]
            print("TASK:", task)
            print("  tools used:", used, "| answer:", out["answer"])
            print()
        print("You built the WHOLE loop from scratch; a REAL model drove 3 tasks and a deterministic guardrail refused the 4th.")
        print("Next: Module 6 hands this exact loop to LangChain's create_agent.")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- You built the **entire loop from scratch** and a **real** `llama3.1:8b` drove it &mdash; chaining tools on the two-step task, using one tool on the others.
- The **deterministic guardrail** refused the *"delete"* request **before any model call** &mdash; safety you can prove, wrapped around an unpredictable model.
- That IS agentic AI. **Module 6** next hands this loop to LangChain's `create_agent`, so you write the tools and let the framework run the loop.

## Your turn (open task &mdash; no grader)
Add a new tool to `TOOLS` (and mention it in `REACT_SYS`) plus a suite task that needs it, and add another
dangerous verb to `DANGEROUS`. Re-run. **What good looks like:** the model uses your new tool on the right
task, the guardrail refuses the new dangerous phrasing, and the step cap always holds.

---
### Key takeaway
You built a guardrailed mini-agent from scratch -- tools, memory, a loop -- and a REAL model drove it over a suite. That IS agentic AI; next, Module 6 hands the loop to LangChain's create_agent.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>